# Spark Data Lake — interactive exploration

This notebook is for **exploring** the data lake and trying ideas out.
It is not part of the pipeline: the production jobs live in `src/spark/`.

Start JupyterLab with `make notebook`, then open the URL it prints.

> Prerequisite: `make pipeline` (or at least `make generate bronze silver`).


## 1. Start the Spark session


In [ ]:
from src.spark.spark_session import get_spark, describe_session
from src.spark.utils import BRONZE, SILVER, GOLD, RAW, human_bytes, path_size
from pyspark.sql import functions as F

spark = get_spark('notebook-exploration')

for k, v in describe_session(spark).items():
    print(f'{k:<22}: {v}')


The Spark UI for this session is at **http://localhost:4040**.
Keep it open: that is where you will see the jobs, the stages and the shuffles.


## 2. Load the three layers


In [ ]:
orders    = spark.read.parquet(str(SILVER / 'orders'))
customers = spark.read.parquet(str(SILVER / 'customers'))
products  = spark.read.parquet(str(SILVER / 'products'))

for name, df in [('orders', orders), ('customers', customers), ('products', products)]:
    print(f'{name:<10} {df.count():>9,} rows | {df.rdd.getNumPartitions()} partitions')


In [ ]:
orders.printSchema()


In [ ]:
orders.show(5, truncate=False)


## 3. Compare the sizes of the layers


In [ ]:
for label, path in [('raw (CSV)', RAW), ('bronze', BRONZE), ('silver', SILVER), ('gold', GOLD)]:
    print(f'{label:<12} {human_bytes(path_size(path)):>10}')


## 4. Revenue by category

`F.broadcast()` on `products`: the table is tiny, so why pay for a shuffle.


In [ ]:
revenue = (orders
      .join(F.broadcast(products), 'product_id')
      .filter(F.col('status').isin(['delivered', 'shipped']))
      .withColumn('revenue', F.col('quantity') * F.col('price'))
      .groupBy('category')
      .agg(F.round(F.sum('revenue'), 2).alias('revenue'),
           F.count('order_id').alias('orders_count'))
      .orderBy(F.col('revenue').desc()))

revenue.show(truncate=False)


### Reading the execution plan

Look for: `BroadcastHashJoin` (a join with no shuffle), `Exchange` (a shuffle),
`PushedFilters` (a filter pushed down into Parquet), `HashAggregate` (the aggregation).


In [ ]:
revenue.explain('formatted')


## 5. Spark SQL: exactly the same thing

Temporary views live for the session and store nothing.


In [ ]:
orders.createOrReplaceTempView('orders')
customers.createOrReplaceTempView('customers')
products.createOrReplaceTempView('products')

spark.sql('''
    SELECT c.country,
           COUNT(*)                            AS orders_count,
           ROUND(SUM(o.quantity * p.price), 2) AS revenue
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN products  p ON o.product_id  = p.product_id
    WHERE o.status IN ('delivered', 'shipped')
    GROUP BY c.country
    ORDER BY revenue DESC
    LIMIT 10
''').show(truncate=False)


## 6. Inspect the partitions

`rows_per_partition` triggers an action: keep it for debugging.


In [ ]:
from src.spark.utils import rows_per_partition

print('Natural        :', rows_per_partition(orders))
print('repartition(8) :', rows_per_partition(orders.repartition(8)))
print('by status      :', rows_per_partition(orders.repartition(F.col('status'))))
print()
print('The last case shows DATA SKEW: only 5 statuses,')
print('and delivered accounts for 62% of the rows.')


## 7. Read the Gold tables


In [ ]:
for table in ['daily_sales', 'product_sales', 'customer_sales', 'country_sales']:
    df = spark.read.parquet(str(GOLD / table))
    print(f'--- {table} ({df.count():,} rows)')
    df.show(3, truncate=False)


## 8. Switch to Pandas to draw a chart

⚠️ `toPandas()` brings **all** the data back to the driver.
On a Gold table (a few thousand rows), no risk at all.
On `orders` (1M rows), it can bring the driver down.
**Always aggregate before converting.**


In [ ]:
import pandas as pd

daily = spark.read.parquet(str(GOLD / 'daily_sales')).toPandas()
daily['date'] = pd.to_datetime(daily['date'])
print(daily.describe())

ax = daily.set_index('date')['total_revenue'].plot(
    figsize=(14, 4), title='Daily revenue')
ax.set_ylabel('Revenue (EUR)')


## 9. Release the resources

Important: as long as the session is alive it holds the worker's cores and
**no other job can start**.


In [ ]:
spark.stop()
